# D1_04 네 가지 파일을 분석 테이블 하나로

> **오늘 질문:** 흩어진 공장 데이터를 어떻게 하나로 모을까?

| 파일 | 형식 | 내용 |
|---|---|---|
| `equipment_master.json` | JSON (중첩) | 설비 10대의 라인·제조사·설치연도 |
| `production_log.csv` | CSV | 생산 기록 (일시, 설비, 제품, 등급) |
| `sensor_log.parquet` | Parquet | 센서 값 (온도, 회전수, 토크, 공구 마모, 진동) |
| `factory.duckdb` | DB 테이블 | `quality_result`: 고장 여부와 고장 유형 |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [1]:
import json
import os

import duckdb
import pandas as pd

DATA = "../../data/bundle/"   # 가상 공장 번들 폴더

## 1. 설비 마스터 (JSON) ✍️

JSON 파일을 열어 읽은 뒤 `pd.json_normalize()`로 표로 펼칩니다.
`sep="_"`를 주면 중첩된 `spec` 아래 값이 `spec_maker`처럼 펼쳐집니다.

> 변수 이름: `equip`

In [2]:
# 힌트: with open(파일, encoding="utf-8") as f: → json.load(f) → pd.json_normalize(데이터, sep="_")
with open(DATA + "equipment_master.json", encoding="utf-8") as f:
    data = json.load(f) # with 절을 사용하면 자동 close 됨
equip = pd.json_normalize(data, sep="_")
print(equip.shape) # 몇행 몇열
equip.head() # 디폴트 5개

(10, 4)


,equip_id,line,spec_maker,spec_install_year
0,MC-01,L1,A사,2019
1,MC-02,L1,A사,2020
2,MC-03,L1,B사,2012
3,MC-04,L1,A사,2021
4,MC-05,L1,C사,2018


## 2. 생산 실적 (CSV) ✍️

CSV는 날짜를 글자로 읽습니다. `parse_dates`로 날짜 컬럼을 지정하세요.

> 변수 이름: `prod`

In [3]:
# 힌트: pd.read_csv(파일, parse_dates=["timestamp"])
prod = pd.read_csv(DATA + "production_log.csv", parse_dates=["timestamp"])
print(prod.shape)
prod.head()

(10015, 5)


,record_id,timestamp,equip_id,product_id,grade
0,1,2025-01-01 00:00:00,MC-08,M14860,M
1,2,2025-01-01 00:14:24,MC-08,L47181,L
2,3,2025-01-01 00:28:48,MC-03,L47182,L
3,4,2025-01-01 00:43:12,MC-01,L47183,L
4,5,2025-01-01 00:57:36,MC-04,L47184,L


## 3. 센서 로그 (Parquet) ✍️

Parquet는 타입을 함께 저장하므로 날짜가 처음부터 날짜로 읽힙니다.

> 변수 이름: `sensor`

In [4]:
# 힌트: pd.read_parquet(파일)
sensor = pd.read_parquet(DATA + "sensor_log.parquet")
print(sensor.shape)
sensor.head()

(10000, 8)


,equip_id,timestamp,air_temp,process_temp,rpm,torque,tool_wear,vibration
0,MC-08,2025-01-01 00:00:00,25.0,35.5,1551,42.8,0,1.84
1,MC-08,2025-01-01 00:14:24,25.1,35.6,1408,46.3,3,2.20
2,MC-03,2025-01-01 00:28:48,25.0,35.4,1498,49.4,5,1.86
3,MC-01,2025-01-01 00:43:12,25.1,35.5,1433,39.5,7,2.51
4,MC-04,2025-01-01 00:57:36,25.1,35.6,1408,40.0,9,2.05


## 4. 품질 결과 (DuckDB) ✍️

DB 파일에 연결하고 SQL로 테이블을 조회합니다. `.df()`를 붙이면 pandas 표로 바뀝니다.

> 변수 이름: `con`

In [5]:
# 힌트: duckdb.connect(파일, read_only=True) → con.sql("SQL").df()
con = duckdb.connect(DATA + "factory.duckdb", read_only=True)
# 검색한다 모든_컬럼 FROM 테이블_명 5개
con.sql("SELECT * FROM quality_result LIMIT 5").df() # 3:05~3:45

,product_id,failure,twf,hdf,pwf,osf,rnf
0,M14860,0,0,0,0,0,0
1,L47181,0,0,0,0,0,0
2,L47182,0,0,0,0,0,0
3,L47183,0,0,0,0,0,0
4,L47184,0,0,0,0,0,0


## 5. 조인 전에 키 확인 📋

생산 실적과 센서 로그는 **설비ID + 일시**로 붙입니다. 두 표의 `timestamp` 타입이 같아야 합니다.

In [6]:
print(prod["timestamp"].dtype, sensor["timestamp"].dtype)

datetime64[ns] datetime64[ns]


## 6. DuckDB로 한 번에 조인 ✍️

DuckDB는 DB 안의 테이블(`quality_result`)과 파이썬 변수에 담긴 표(`prod`, `sensor`, `equip`)를 함께 SQL로 조인할 수 있습니다.

| 붙이는 표 | 키 |
|---|---|
| 센서 로그 `s` | 설비ID + 일시 |
| 품질 결과 `q` | 제품ID |
| 설비 마스터 `e` | 설비ID |

> 변수 이름: `sql`, `df`

In [7]:
# 힌트: LEFT JOIN 표 별칭 ON 키 = 키, 컬럼 AS 새이름
sql = """
SELECT p.*,
       s.air_temp, s.process_temp, s.rpm, s.torque, s.tool_wear, s.vibration,
       q.failure, q.twf, q.hdf, q.pwf, q.osf, q.rnf,
       e.line, e.spec_maker AS maker, e.spec_install_year AS install_year
FROM prod p
LEFT JOIN sensor s ON p.equip_id = s.equip_id AND p.timestamp = s.timestamp
LEFT JOIN quality_result q ON p.product_id = q.product_id
LEFT JOIN equip e ON p.equip_id = e.equip_id
"""
df = con.sql(sql).df()

## 7. 조인 결과 확인 ✍️

조인 후에는 **행 수가 그대로인지** 꼭 확인합니다. 키가 어긋나면 행이 늘거나, 값이 비게 됩니다.

In [8]:
print(f"생산 실적 {len(prod)} 행 -> 분석 테이블 {len(df)} 행")
print(f"센서 값이 안 붙은 행: {df['air_temp'].isna().sum()}")

생산 실적 10015 행 -> 분석 테이블 10015 행
센서 값이 안 붙은 행: 0


**Q1.** 생산 기록은 10,000건이라고 들었는데 행 수는 몇 개인가요? 왜 그럴까요?

> 답: 10,015행 맞음. 중복 저장이 의심 --> 5교시 품질 점검에서 확인


In [9]:
df.head()

,record_id,timestamp,equip_id,product_id,grade,air_temp,process_temp,rpm,torque,tool_wear,vibration,failure,twf,hdf,pwf,osf,rnf,line,maker,install_year
0,1,2025-01-01 00:00:00,MC-08,M14860,M,25.0,35.5,1551,42.8,0,1.84,0,0,0,0,0,0,L2,B사,2013
1,2,2025-01-01 00:14:24,MC-08,L47181,L,25.1,35.6,1408,46.3,3,2.20,0,0,0,0,0,0,L2,B사,2013
2,3,2025-01-01 00:28:48,MC-03,L47182,L,25.0,35.4,1498,49.4,5,1.86,0,0,0,0,0,0,L1,B사,2012
3,4,2025-01-01 00:43:12,MC-01,L47183,L,25.1,35.5,1433,39.5,7,2.51,0,0,0,0,0,0,L1,A사,2019
4,5,2025-01-01 00:57:36,MC-04,L47184,L,25.1,35.6,1408,40.0,9,2.05,0,0,0,0,0,0,L1,A사,2021


## 8. CSV vs Parquet 저장 비교 ✍️

같은 분석 테이블을 두 형식으로 저장해 크기와 읽는 속도를 비교합니다.

In [10]:
# 힌트: 표.to_csv(파일, index=False), 표.to_parquet(파일, index=False)
df.to_csv("analysis_table.csv", index=False) # 디폴트가 index 포함 저장, False 빼고 저장
df.to_parquet("analysis_table.parquet", index=False)

In [11]:
csv_kb = os.path.getsize("analysis_table.csv") / 1024
pq_kb = os.path.getsize("analysis_table.parquet") / 1024
print(f"CSV     {csv_kb:,.0f} KB")
print(f"Parquet {pq_kb:,.0f} KB  (CSV의 {pq_kb / csv_kb:.0%})")

CSV     922 KB
Parquet 289 KB  (CSV의 31%)


`%timeit`은 같은 코드를 여러 번 실행해 평균 시간을 잽니다.

In [12]:
# 힌트: %timeit -n 5 코드
%timeit -n 5 pd.read_csv("analysis_table.csv")
%timeit -n 5 pd.read_parquet("analysis_table.parquet")

25.7 ms ± 3.43 ms per loop (mean ± std. dev. of 7 runs, 5 loops each)
7.67 ms ± 764 μs per loop (mean ± std. dev. of 7 runs, 5 loops each)


## 정리 💬

**Q2.** 매일 쌓이는 센서 로그를 분석용으로 저장한다면 어떤 형식을 고르겠습니까? 이유는?

> 답: Pqrquet. 용량도 작고 읽는 속도 빠름.


In [13]:
# end